# Laboratorio 1 — Ajuste del Movimiento Armónico Simple (MAS)

Notebook adaptado a los datos experimentales del grupo para 10 Hz, 20 Hz y 30 Hz.

El cargador se detiene al terminar el bloque experimental de 7 columnas, porque los archivos `.txt` contienen después una sección adicional de dos columnas que no forma parte de la medición principal.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams["figure.figsize"] = (10, 4)

ARCHIVOS = {
    "10 Hz": "../datos/Biela-Manivela-10Hz-20s (negativo angulo).txt",
    "20 Hz": "../datos/Biela-Manivela-20Hz-20s.txt",
    "30 Hz": "../datos/Biela-Manivela-30Hz-20s.txt",
}

INTERVALOS = {
    "10 Hz": (5, 15),
    "20 Hz": (5, 15),
    "30 Hz": (8, 16),
}

COLUMNAS = ["t", "theta", "omega", "alpha", "x", "v", "a"]

def cargar_loggerpro(ruta):
    filas = []
    with open(ruta, "r", encoding="utf-8-sig") as f:
        lineas = f.readlines()[7:]
    for linea in lineas:
        partes = linea.rstrip("\n").split("\t")
        if len(partes) != 7:
            break
        try:
            filas.append([float(v) for v in partes])
        except ValueError:
            break
    return pd.DataFrame(filas, columns=COLUMNAS)

datos = {nombre: cargar_loggerpro(ruta) for nombre, ruta in ARCHIVOS.items()}

for nombre, df in datos.items():
    print(f"{nombre}: {len(df)} muestras | t = {df.t.iloc[0]:.3f}–{df.t.iloc[-1]:.3f} s | Δt ≈ {np.median(np.diff(df.t)):.6f} s")


## 1. Ajuste no lineal del MAS

Se utilizan las expresiones indicadas en la guía:

x(t) = A0 + A cos(ωt + φ)

v(t) = -ω A sin(ωt + φ)

a(t) = -ω² A cos(ωt + φ)


In [ ]:
def x_mas(t, A0, A, w, phi):
    return A0 + A*np.cos(w*t + phi)

def v_mas(t, A, w, phi):
    return -w*A*np.sin(w*t + phi)

def a_mas(t, A, w, phi):
    return -w**2*A*np.cos(w*t + phi)

def r2(y, yhat):
    return 1 - np.sum((y-yhat)**2)/np.sum((y-np.mean(y))**2)

resultados_mas = {}

for nombre, df in datos.items():
    tmin, tmax = INTERVALOS[nombre]
    d = df[(df.t >= tmin) & (df.t <= tmax)]
    t, x, v, a = d.t.to_numpy(), d.x.to_numpy(), d.v.to_numpy(), d.a.to_numpy()

    A0_est = (x.max()+x.min())/2
    A_est = (x.max()-x.min())/2

    popt_x, _ = curve_fit(
        x_mas, t, x,
        p0=[A0_est, A_est, -5.0, 0],
        maxfev=20000
    )
    A0_fit, A_fit, w_fit, phi_fit = popt_x

    popt_v, _ = curve_fit(v_mas, t, v, p0=[abs(A_fit), w_fit, phi_fit], maxfev=20000)
    popt_a, _ = curve_fit(a_mas, t, a, p0=[abs(A_fit), w_fit, phi_fit], maxfev=20000)

    xhat = x_mas(t, *popt_x)
    vhat = v_mas(t, *popt_v)
    ahat = a_mas(t, *popt_a)

    resultados_mas[nombre] = {
        "A0": A0_fit, "A": A_fit, "omega": w_fit, "phi": phi_fit,
        "R2_x": r2(x, xhat), "R2_v": r2(v, vhat), "R2_a": r2(a, ahat)
    }

    print(f"\n{nombre}: intervalo {tmin}–{tmax} s")
    print(f"A0 = {A0_fit:.6f} m | A = {A_fit:.6f} m | ω = {w_fit:.6f} rad/s | φ = {phi_fit:.6f} rad")
    print(f"R²(x) = {resultados_mas[nombre]['R2_x']:.4f} | R²(v) = {resultados_mas[nombre]['R2_v']:.4f} | R²(a) = {resultados_mas[nombre]['R2_a']:.4f}")

    fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

    axs[0].plot(t, x, "o", ms=3, label="Datos")
    axs[0].plot(t, xhat, label="Ajuste MAS")
    axs[0].set_ylabel("x (m)"); axs[0].legend(); axs[0].grid(True)

    axs[1].plot(t, v, "o", ms=3, label="Datos")
    axs[1].plot(t, vhat, label="Ajuste MAS")
    axs[1].set_ylabel("v (m/s)"); axs[1].legend(); axs[1].grid(True)

    axs[2].plot(t, a, "o", ms=3, label="Datos")
    axs[2].plot(t, ahat, label="Ajuste MAS")
    axs[2].set_ylabel("a (m/s²)"); axs[2].set_xlabel("Tiempo (s)")
    axs[2].legend(); axs[2].grid(True)

    fig.suptitle(f"Ajuste MAS — {nombre}")
    plt.tight_layout()
    plt.show()

tabla_mas = pd.DataFrame(resultados_mas).T
tabla_mas.index.name = "Muestreo"
display(tabla_mas)


### Observación

El ajuste MAS es una aproximación. La guía explica que la longitud finita de la biela introduce términos anarmónicos, por lo que las diferencias entre los datos experimentales y el modelo deben discutirse físicamente.
